# Chapter 1: Chunking

**Estimated time: 15 minutes**

Your RAG system returns the wrong answer and you blame the model. But the real problem happened before any search ran. You chunked your documents wrong.

In this notebook you will split the same long employee handbook two different ways and watch the same question go from a weak retrieval with a vague answer to a confident retrieval with one clean sentence. The only thing that changes is one number. Chunk size.

Run every cell top to bottom. When you reach the "Try it yourself" section, change the numbers and rerun.

## What you will see in three minutes

You are going to ask this question:

> How many weeks of paid parental leave does SkillAgents AI offer for a primary caregiver, and when does it have to be taken?

The answer lives in one specific section of one specific document, the SkillAgents employee handbook. The handbook is roughly nine thousand characters long and covers ten different HR topics. Parental leave is one short section inside all of that.

You will run the question through two versions of the same pipeline.

- **Version A** chunks each document into 12000 character blocks. This is what happens when someone reaches for "give the model maximum context". The whole handbook becomes one giant chunk that averages parental leave, offices, hiring, equipment, travel, speaking, communication norms, performance reviews, sabbaticals, and off-boarding into one embedding. The query similarity lands around 0.57, which is weak, and the LLM answer hedges.
- **Version B** chunks the same documents into 400 character blocks. Now the parental leave section becomes its own isolated chunk. The query similarity climbs to around 0.86, which is strong, and the LLM answer is one clean sentence with the exact eighteen-week figure and the twelve-month window.

Same documents. Same embedding model. Same LLM. Same question. One number changes on the splitter. The retrieval score jumps by about 0.29 and the answer style flips from vague to production ready.

Because this is the foundation chapter, every RAG knob lives on the page here instead of behind a helper. You will see the splitter class, the embedding model, and the distance strategy out in the open so that you can swap any of them later.

Now let us get set up.

## One time setup, getting your API keys

This is the only chapter that walks through API keys. Every other chapter assumes the keys are already in Colab secrets and picks them up automatically.

You need three keys. All three have free tiers that cover the entire series. Total cost across all seven chapters is under two dollars.

1. **OpenAI** at [platform.openai.com](https://platform.openai.com). New accounts get five dollars of free credit. Copy your key.
2. **LangSmith** at [smith.langchain.com](https://smith.langchain.com). Sign up, go to Settings, API Keys, create a personal key. Free tier gives you five thousand traces per month.
3. **Cohere** at [dashboard.cohere.com](https://dashboard.cohere.com). Sign up, go to API Keys, copy your trial key. Free tier gives you one thousand rerank calls per month. You will not need this until Chapter 5, but add it now so every notebook just works.

Now add them to Colab secrets. Click the **key icon** on the left sidebar of Colab. Click **+ Add new secret** three times and add:

- `OPENAI_API_KEY` set to your OpenAI key
- `LANGCHAIN_API_KEY` set to your LangSmith key
- `COHERE_API_KEY` set to your Cohere key

For each one, toggle **Notebook access** on. You only do this once. Every chapter from now on picks the keys up automatically.

## Install the dependencies

Run the next cell once. It installs the Python packages the notebook needs.

If you are running this in Google Colab, the cell detects Colab and pip installs everything fresh. If you are running locally in Jupyter, the cell skips the install (it assumes you already ran `pip install -r requirements.txt` in your virtual environment). Either way, you just click Run and keep going.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 60 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without this
    # a stale repo from a prior session would keep running the old utils code.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. If a prior run already imported
# an older version of utils.shared, Python will keep returning that stale copy
# even after we replace the file on disk. Clear the cache so the next import
# reads the fresh file.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your three API keys out of Colab secrets and turned on LangSmith tracing. From now on every embedding call, every chat call, and every retrieval step gets logged automatically to a clickable waterfall trace at [smith.langchain.com](https://smith.langchain.com).

## Look at the corpus

You are about to load the SkillAgents AI corpus. The star of this chapter is `ch1_skillagents_handbook.md`, the internal employee handbook. The other documents are there for later chapters (pricing, refund policy, product guide, billing FAQ, error codes, outdated blog post, a short refund-quick-answers file), and they also give the retriever realistic noise to compete against.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()

for d in docs[:6]:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## Peek at the parental leave section

Before we run anything, let us look at the exact text that answers the question. It lives in the handbook under the "Parental leave and caregiving leave" section. This section is about 650 characters long, which you should remember because it comes back in a minute.

In [ ]:
handbook = next(d for d in docs if d.metadata["source"] == "ch1_skillagents_handbook.md")

start = handbook.page_content.find("Parental leave and caregiving leave")
print(handbook.page_content[start:start + 700])

This is the target. When we run the question through the pipeline, we want the top retrieved chunk to be this exact passage. Anything else is a miss.

Notice how short it is. One focused section, eighteen weeks primary, six weeks secondary, a twelve-month window. Roughly six hundred fifty characters of specific answer surrounded by nine thousand characters of unrelated handbook content. Chunk size decides whether that short focused section gets its own chunk or gets averaged into a mega-chunk of HR trivia.

## What your chunks actually look like

Before we run retrieval, let us look at what the splitter actually produces. We are going to use `RecursiveCharacterTextSplitter` with `chunk_size=400` and `chunk_overlap=50`, then peek at the first five chunks from the whole corpus. This is the single most useful debug view in any RAG system and you will rarely see it on a product demo.

Notice that we are importing `RecursiveCharacterTextSplitter` directly in the cell below, not calling a hidden helper. Chapter 1 is the foundation chapter, so every knob sits on the page. You see the class name, you see the parameters, you see the result.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.shared import pretty_print_chunks

# The splitter you are about to use for the rest of this chapter.
# RecursiveCharacterTextSplitter is the LangChain default for free-form text.
# It tries paragraph breaks first, then sentence breaks, then word breaks,
# and only falls back to mid-word as a last resort.
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

sample_chunks = splitter.split_documents(docs)
print(f"Total chunks from all the documents: {len(sample_chunks)}")
pretty_print_chunks(sample_chunks, n=5)

Look at the length column. Each chunk is close to 500 characters but not exactly. The splitter respects sentence and paragraph boundaries where it can, so some chunks are 420 characters, others are 510. The preview column shows the actual text of each chunk. This is what your embedding model is going to see, one chunk at a time.

## The splitter you just saw, and the others LangChain ships

The splitter in the cell above is `RecursiveCharacterTextSplitter`. It is the right default for free form text, and it is the one you will use for the rest of this chapter. LangChain ships three other splitters that are worth knowing by name right now, before you build anything of your own.

- **`MarkdownHeaderTextSplitter`** splits a markdown document on its headers (`#`, `##`, `###`). If your source is a knowledge base with clear section structure (like our `product_guide.md`), this produces one chunk per section, which never cuts a topic in half.
- **`TokenTextSplitter`** splits by token count instead of character count. Use this when you need to respect an exact token budget for a specific embedding model, for example keeping every chunk under 512 tokens.
- **`SemanticChunker`** from `langchain_experimental` splits at semantic boundaries by computing embedding similarity between adjacent sentences. Slower and more expensive, but produces high quality chunks for narrative content like podcast transcripts or long essays.

Which one to pick depends on what your content looks like, not on which one is "best". For a PDF of a contract, use `RecursiveCharacterTextSplitter`. For a markdown knowledge base, use `MarkdownHeaderTextSplitter`. For long narratives, try `SemanticChunker`. The next two cells stick with the recursive splitter on our corpus.

## Version A: the default from the quickstart

Most RAG quickstart tutorials land on one of two reasonable-sounding defaults. Some suggest 1000 to 2000 characters. Others say "just give the model a lot of context, modern LLMs can handle it" and pick something huge like 12000. We are going to start with the second one, because it is the failure mode that ships to production more often than people admit.

The next cell constructs three things by hand: the splitter, the embedding model, and the FAISS index with its distance strategy. These are the three knobs that sit inside every RAG pipeline in the world. You do not have to memorize them, but you should see them at least once with the class names on the page.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from utils.shared import search, show_results

# Knob 1: the splitter. RecursiveCharacterTextSplitter is the LangChain
# default for free-form text. Swap to MarkdownHeaderTextSplitter for a
# markdown knowledge base, or TokenTextSplitter for an exact token budget.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=12000,   # "Give the model maximum context" default. Watch what this does.
    chunk_overlap=0,
)

# Knob 2: the embedding model. text-embedding-3-small returns 1536-dimensional
# unit vectors and costs about two cents per million tokens. Swap to
# text-embedding-3-large for higher quality, or a local model via Ollama to
# keep data on your own machine.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chunks_v_a = splitter.split_documents(docs)

# Knob 3: the distance strategy. MAX_INNER_PRODUCT on unit vectors is
# mathematically identical to cosine similarity in the range 0 to 1, where 1
# is a perfect semantic match. Swap to EUCLIDEAN_DISTANCE for non-normalized
# embeddings, or to COSINE if you want FAISS to normalize for you (slower).
default_index = FAISS.from_documents(
    documents=chunks_v_a,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)
print(f"Version A: {len(chunks_v_a)} chunks at chunk_size=12000")

question = "How many weeks of paid parental leave does SkillAgents AI offer for a primary caregiver, and when does it have to be taken?"

default_results = search(default_index, question, k=3)
show_results(default_results, question=question)

## What went wrong

The top row in the table probably points at `ch1_skillagents_handbook.md`. Good, the retriever found the right document. But look at the similarity column.

**Reading cosine similarity scores.** Cosine similarity runs from 0 to 1. Higher is better. 1.0 means the query and the chunk are identical in meaning. 0 means they share no semantic overlap. In this notebook the useful range runs from about 0.4 for a weak match to about 0.85 for a direct semantic match. Anything under 0.6 is a weak signal. Our top score here is around 0.57. Weak.

Why is the score so weak? Because the 12000 character chunk contains the entire handbook as a single block. Parental leave is roughly 650 characters of that, which is around seven percent of the chunk. The other ninety-three percent is offices, hiring, equipment, travel, speaking, communication norms, performance reviews, sabbaticals, and off-boarding. The embedding averages all ten topics into one 1536-dimensional vector. The query about parental leave pulls weakly on that average because parental leave content is a minority of the input.

An LLM given this chunk has to find parental leave inside 12000 characters of unrelated HR text and then state the exact numbers. It might do it. It might hedge. It might mix in information from the other nine sections that the student never asked about.

## Version B: paragraph-sized chunks

Same three knobs. Same data. Same embedding model. Same distance strategy. The only thing that changes is the `chunk_size` argument on the splitter. Watch what happens to the similarity score.

In [ ]:
# Same splitter class, same embedding model, same distance strategy.
# Only chunk_size and chunk_overlap change.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chunks_v_b = splitter.split_documents(docs)

focused_index = FAISS.from_documents(
    documents=chunks_v_b,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)
print(f"Version B: {len(chunks_v_b)} chunks at chunk_size=400")

focused_results = search(focused_index, question, k=3)
show_results(focused_results, question=question)

## What got better

Look at the top score. It should climb from around 0.57 to around 0.86. That is a jump of roughly 0.30, and it is the difference between "the embedding model is guessing which part of the handbook you meant" and "the embedding model is certain this chunk is the answer". The preview column shows the exact parental leave section with almost nothing else in the way.

Same document. Same embedding model. Same distance strategy. Different chunk size. The splitter call is the single thing that changed, and the retrieval quality moved two steps up the gradient.

Now let us see if this actually changes the LLM's answer.

## Why chunk overlap matters

You may have noticed `chunk_overlap=50` in the code above. That means adjacent chunks share 50 characters of boundary text. Here is why that matters.

Imagine a sentence that happens to land right at character 498 of a chunk boundary. With zero overlap, the first half of the sentence lives in chunk N and the second half lives in chunk N+1. Neither chunk contains the complete sentence. A retriever searching for that sentence gets a weaker match from both, because both are incomplete.

Overlap fixes this by letting adjacent chunks share a small window of text, so any sentence near a boundary appears complete in at least one chunk.

In [ ]:
# Same splitter class. Only the overlap changes.
splitter_no_overlap = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0)
splitter_with_overlap = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)

chunks_no_overlap = splitter_no_overlap.split_documents(docs)
chunks_with_overlap = splitter_with_overlap.split_documents(docs)

print(f"Chunks with overlap=0:  {len(chunks_no_overlap)}")
print(f"Chunks with overlap=80: {len(chunks_with_overlap)}")

Overlap adds a few more chunks, which means slightly more embedding cost at index time and slightly more storage. The benefit is that you do not lose matches at chunk boundaries. For most production RAG systems, a `chunk_overlap` of ten to twenty percent of `chunk_size` is a reasonable default. That is why we used fifty on a five hundred character chunk.

## The moment that matters, what the LLM actually says

So far we have been looking at similarity scores. A PM does not care about similarity scores. A PM cares about the answer the customer sees. Let us feed both sets of chunks to `gpt-4o-mini` and compare.

In [ ]:
from utils.shared import generate_answer

answer_default = generate_answer(default_results, question)
answer_focused = generate_answer(focused_results, question)

print("VERSION A (chunk_size=12000)")
print("-" * 60)
print(answer_default)
print()
print("VERSION B (chunk_size=400)")
print("-" * 60)
print(answer_focused)

## Read both answers carefully

Both answers should include the right numbers. gpt-4o-mini is strong enough to extract eighteen weeks and the twelve-month window even from the big mixed handbook chunk that Version A fed it. So this chapter is not a "Version A is wrong, Version B is right" lesson. The retrieval tables above told the story more honestly than the answers will.

Here is the real comparison, the thing a PM should internalize.

Version A retrieved the answer chunk at a similarity score around 0.57, which is weak. That chunk was the entire handbook averaged into one embedding. The LLM got lucky because the model is good and the facts were present in the raw text. On a messier document, or with a question that required more reasoning across the chunk, Version A would have gotten lost in the mixed content.

Version B retrieved the answer chunk at a similarity score around 0.86, which is strong. The chunk the LLM received was just the parental leave section, no other HR topics mixed in. The retrieval was deterministic instead of lucky. In production, deterministic retrieval is the whole point. It survives edge cases, new questions, messier docs, and longer corpora.

The retrieval layer is the single biggest lever on answer quality, and chunk size is the single biggest lever on the retrieval layer. Chapter 1 is the chapter where you learn to watch the similarity score, not just the final answer. Cleaner retrieval is a better product even when the LLM is smart enough to paper over the gap, because you cannot always count on the LLM to paper over the gap.

## Open your LangSmith trace

If you want to see every step inside the pipeline, open [smith.langchain.com](https://smith.langchain.com), click into the project called `rag-for-pms`, and open the most recent trace. You will see two retrieval calls and two generation calls, side by side, with every chunk, every score, every token count, and every millisecond of latency.

Optional but worth five minutes.

## What you can do at work on Monday

You do not need to remember chunk size numbers. You need to remember the question.

When you review a RAG feature your team is building, ask three things.

1. What chunk size did you pick and why?
2. Show me the top three retrieved chunks for a real user question.
3. What happens if the top chunk is the wrong section of the right document?

If the team cannot answer these cleanly, the product will fail in production. Most RAG failures live in the retrieval layer, not the LLM, and chunk size is the single biggest lever in the retrieval layer.

## Try it yourself

Pick any of these. Change the number in the cell above, rerun, watch what happens.

1. Change `chunk_size` in the Version B cell to **200**. Rerun the search and the LLM answer cells. Watch the top score climb even higher and the preview column shrink to one or two sentences.
2. Change `chunk_size` in the Version A cell to **20000**. Rerun the search. Nothing changes because the entire handbook is already one chunk at 12000. The splitter cannot split a doc that fits in one chunk. This is an important failure mode to see at least once.
3. Change the `question` variable to `"What is the sabbatical policy at SkillAgents?"`. Rerun the Version B cells. Watch the top match switch from the parental leave section to the sabbatical section without changing any other code. Different question, same pipeline, different chunk.

## Homework

Two exercises you can do on your own. Each one takes about fifteen minutes.

1. **Try `MarkdownHeaderTextSplitter`.** Look it up in the LangChain docs. Use it on `data/skillagents/product_guide.md` to split by H2 headers instead of character count. Print the first three chunks it produces. Do the chunks look cleaner or messier than the default splitter?

2. **Apply it to your own company.** Pick one real document from your own product. A refund policy, a pricing page, a feature guide. Drop it into `data/skillagents/` alongside the others. Rerun the notebook with a real question from a real user. Does the retrieval work? If not, what chunk size would you try next? Share your before-and-after in the cohort Slack.

## What is next

In Chapter 2 your chunking is perfect, your index is clean, and your vector search still returns the wrong document. Why? Because the user asked a vague question and your pipeline searched for the exact wrong thing.

You are going to fix this by rewriting the question before it ever touches the vector store. The technique is called query translation, and it is the single biggest fix you can make on a production RAG system.

See you in Chapter 2.